# 01. Check the connection

Proves that Spark, the Iceberg REST catalog and MinIO work together by writing and reading a tiny table.

## What to look at

- The **settings table**: Spark reaches MinIO at `localhost:9000` with path-style access and an explicit region.
- The **namespaces**: `bronze`, `silver`, `gold`.
- The script log (collapsed) ends in `PASS` after the data file path starts with `s3://`.

In [ ]:
import sys, pathlib
_nb = next(p for p in (pathlib.Path.cwd(), pathlib.Path.cwd() / "notebooks") if (p / "_setup.py").exists())
sys.path.insert(0, str(_nb))
from _setup import *   # puts src/ on the path, loads .env, gives run_script / get_spark / sql / minio helpers
print("SAMPLE_ROWS =", common.SAMPLE_ROWS, "(None = all rows)")

In [ ]:
pd.DataFrame([
    ("catalog name", common.CATALOG), ("catalog type", "rest"), ("REST URI", common.CATALOG_URI),
    ("S3 endpoint (client side)", common.S3_ENDPOINT), ("path-style access", "true"),
    ("region", common.AWS_REGION), ("warehouse bucket", common.WAREHOUSE_BUCKET),
    ("driver memory", common.env("SPARK_DRIVER_MEMORY", "4g")),
], columns=["setting", "value"])

In [ ]:
run_script("01_check_connection.py")

In [ ]:
spark = get_spark("nb01")
sql(spark, f"SHOW NAMESPACES IN {common.CATALOG}")

In [ ]:
# the check script drops its tiny table (PURGE), so the bucket holds no leftovers from it
leftover = minio_objects("bronze/connection_check/")
print("objects left from the check table:", len(leftover))

In [ ]:
# Cleanup. Skip this cell while you are showing the Spark UI (http://localhost:4040): it closes with the session.
stop_spark()

## Open MinIO now

http://localhost:9001 (login `minioadmin` / `minioadmin`), bucket **warehouse**.

Nothing new should remain: the check script writes a tiny table (`bronze/connection_check/`) and then drops it with PURGE, so its files are gone. Namespaces live in the catalog, not in S3. (To see the file appear, run the script from a terminal and refresh MinIO while it runs.)